# Team MLB · SVM 추가 실험 1: 특징 그룹 단독 학습 (Group-only)

g1~g6 중 **한 그룹만으로** 학습했을 때 웹사이트를 얼마나 구별하는지 본다. 기존 그룹 제거(ablation) 결과와 함께 보면 그룹 간 정보 중복을 설명할 수 있다.

**공정 비교 조건 (그룹 제거 실험과 동일)**
- 분할: `split_seed42.csv`의 학습용 기록만 사용, `StratifiedKFold(5, shuffle=True, random_state=42)`
- 전처리: g1~g5 `log1p` → 학습 구간 기준 상수 특징 제거 → 표준화 (fold마다 학습 구간에서만 적합)
- 모델 설정: 그룹 제거 실험과 같은 값으로 **고정**하고 그룹마다 다시 고르지 않는다
  - Closed-world: (log2C, log2γ) = (6, −6) — 전체 109개 기준 Closed-world 최적값
  - Open-world 이진: (log2C, log2γ) = (3, −4) — 전체 109개 기준 이진 최적값
- 지표: 5-fold CV 정확도 평균 ± 표준편차. **최종 평가(test) 기록은 사용하지 않는다.**

| 순서 | 내용 |
| --- | --- |
| 0~3 | 설정·로드·분할·공통 함수 (MLB_SVM 노트북과 동일) |
| 4 | Group-only 함수 |
| 5 | Closed-world (95 클래스) |
| 6 | Open-world 이진 |
| 7 | 그룹 제거 결과와 나란히 보기 |
| 8 | 저장 |

## 0. 설정

In [1]:
FEATURE_DIR = "/content/drive/MyDrive/MLB-Project/feature"
RESULT_DIR  = "/content/drive/MyDrive/MLB-Project/results/svm"
SPLIT_PATH  = "/content/drive/MyDrive/MLB-Project/results/split_seed42.csv"   # LR과 같은 분할 파일
UNMON_NAME  = "unmon10k"
TRIM        = 0
TRIMS_TO_COMPARE = [0, 10, 20]

SEED     = 42
N_FOLDS  = 5
N_JOBS   = -1

# C·gamma 탐색: log2 단위. 1차는 2칸 간격, 최적값이 경계면 범위를 넓히고, 마지막에 1칸 간격으로 주변을 세밀 탐색
LOG2_C_COARSE = [1, 3, 5, 7, 9, 11]
LOG2_G_COARSE = [-11, -9, -7, -5, -3, -1]

RUN_ABLATION = True    # 9번 그룹 제거 실험

In [2]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(RESULT_DIR, exist_ok=True)
print("FEATURE_DIR:", FEATURE_DIR)
print("RESULT_DIR :", RESULT_DIR)
print("SPLIT_PATH :", SPLIT_PATH)

FEATURE_DIR: /content/drive/MyDrive/MLB-Project/feature
RESULT_DIR : /content/drive/MyDrive/MLB-Project/results/svm
SPLIT_PATH : /content/drive/MyDrive/MLB-Project/feature/split_seed42.csv


In [3]:
import time, itertools
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 1. CSV 로드 · 점검

- 열은 정보 6개 + 특징 110개여야 한다. NaN·무한대가 있으면 데이터팀에 알린다.
- `g5_iat_pos_max`와 `g5_iat_all_max`는 모든 기록에서 값이 같다(데이터팀 문서 8-4). SVM에서는 같은 정보가 거리 계산에 두 번 들어가므로 `g5_iat_pos_max`를 뺀다. **SVM의 "전체 특징"은 109개**다.
- log 변환 대상: 0 이상이고 분포가 한쪽으로 긴 g1~g5 특징. 시간 간격 사분위수는 왜도가 100을 넘는다. g6 누적값은 음수가 있어 제외한다.

In [4]:
INFO_COLS = ["record_id", "dataset", "page_id", "sample_idx", "label", "trim"]
DUP_DROP  = ["g5_iat_pos_max"]

def load_features(kind, trim):
    path = os.path.join(FEATURE_DIR, f"features_{kind}_trim{trim}.csv")
    df = pd.read_csv(path)
    feat = [c for c in df.columns if c[:3] in ("g1_", "g2_", "g3_", "g4_", "g5_", "g6_")]
    assert list(df.columns[:6]) == INFO_COLS, f"정보 열 순서가 다름: {path}"
    assert len(feat) == 110, f"특징 수 {len(feat)}개: {path}"
    X = df[feat].to_numpy(dtype=float)
    print(f"{os.path.basename(path)}: {df.shape}, NaN {int(np.isnan(X).sum())}, inf {int(np.isinf(X).sum())}")
    assert (df[[c for c in feat if not c.startswith("g6_")]] >= 0).all().all(), "g1~g5에 음수가 있음 → log 변환 규칙 확인"
    return df

mon, unmon = load_features("mon", TRIM), load_features(UNMON_NAME, TRIM)
FEATS_ALL = [c for c in mon.columns if c[:3] in ("g1_", "g2_", "g3_", "g4_", "g5_", "g6_") and c not in DUP_DROP]
GROUPS = {g: [c for c in FEATS_ALL if c.startswith(g + "_")] for g in ["g1", "g2", "g3", "g4", "g5", "g6"]}
LOG_COLS = set(c for c in FEATS_ALL if not c.startswith("g6_"))
print("SVM 전체 특징:", len(FEATS_ALL), {g: len(v) for g, v in GROUPS.items()})

skew = mon[FEATS_ALL].skew().sort_values(ascending=False)
print("왜도 상위 5개 (log 변환 전):"); print(skew.head().round(1).to_string())
const_mon = [c for c in FEATS_ALL if mon[c].nunique() == 1]
print("mon trim0에서 상수인 특징:", const_mon, "→ 파이프라인의 VarianceThreshold가 학습 구간 기준으로 제거")

features_mon_trim0.csv: (19000, 116), NaN 0, inf 0
features_unmon10k_trim0.csv: (10000, 116), NaN 0, inf 0
SVM 전체 특징: 109 {'g1': 3, 'g2': 17, 'g3': 6, 'g4': 3, 'g5': 30, 'g6': 50}
왜도 상위 5개 (log 변환 전):
g5_iat_all_q50    137.8
g5_iat_in_q50     101.3
g5_iat_out_q25     22.3
g5_iat_out_q75     18.5
g5_iat_in_q75      17.2
mon trim0에서 상수인 특징: ['g5_iat_all_q25', 'g5_iat_in_q25', 'g6_cumul_00'] → 파이프라인의 VarianceThreshold가 학습 구간 기준으로 제거


## 2. 학습 / 최종 평가 분할 (모델팀 공유)

LR 노트북이 만든 `split_seed42.csv`를 그대로 읽는다. `record_id` 기준이라 trim0/10/20 모두 같은 분할이 적용된다.
최종 평가 기록은 특징 구성·C·gamma 선택에 쓰지 않는다.

In [5]:
split = pd.read_csv(SPLIT_PATH)
SPLIT = dict(zip(split.record_id, split.split))

def attach_split(df):
    s = df.record_id.map(SPLIT)
    assert s.notna().all(), "분할 파일에 없는 record_id가 있음"
    df = df.copy(); df["split"] = s.to_numpy()
    return df

mon, unmon = attach_split(mon), attach_split(unmon)
_all = pd.concat([mon, unmon], ignore_index=True)
print(pd.crosstab(_all.dataset, _all.split))

split     test  train
dataset              
mon       3800  15200
unmon10k  2000   8000


## 3. 공통 함수

**파이프라인** (5-fold의 각 학습 구간에서만 적합해 정보 유출을 막는다)
1. g1~g5 특징에 `log1p` 적용 (g6 누적값은 그대로)
2. `VarianceThreshold`: 학습 구간에서 상수인 특징 제거 (trim·데이터마다 상수 특징이 다르다)
3. `StandardScaler`
4. `SVC(kernel="rbf")`, 다중 분류는 one-vs-one

**C·gamma 탐색** (`tune`)
- 1차: log2 단위 2칸 간격 격자를 5-fold로 평가한다.
- 최적값이 탐색 범위의 끝이면 그 방향으로 2칸씩 넓혀 다시 평가한다(최대 2번).
- 2차: 최적값 주변을 1칸 간격으로 세밀하게 평가한다.
- `fixed=(log2C, log2gamma)`를 주면 탐색 없이 그 설정의 5-fold 점수만 구한다(같은 설정으로 조건을 비교하는 7·9번 실험).

Open-world 이진의 ROC·PR 곡선은 `decision_function` 점수로 계산한다. 확률 추정(`probability=True`)은 학습 시간이 크게 늘고 이진 곡선에는 필요하지 않아 쓰지 않는다. 판정 기준은 LR(확률 0.5)과 맞춰 SVM 기본 기준(점수 0)을 쓴다.

In [6]:
def safe_log1p(X):
    return np.log1p(np.clip(X, 0, None))

def make_svm(feats, C=1.0, gamma="scale", class_weight=None):
    log_cols = [c for c in feats if c in LOG_COLS]
    pre = ColumnTransformer([("log", FunctionTransformer(safe_log1p, feature_names_out="one-to-one"), log_cols)],
                            remainder="passthrough", verbose_feature_names_out=False)
    return Pipeline([("pre", pre), ("var", VarianceThreshold()), ("sc", StandardScaler()),
                     ("svc", SVC(kernel="rbf", C=C, gamma=gamma, class_weight=class_weight, cache_size=1000))])

CV = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
CV_TABLES = {}

def _grid_eval(X, y, pairs, class_weight):
    grid = [{"svc__C": [2.0 ** c], "svc__gamma": [2.0 ** g]} for c, g in pairs]
    gs = GridSearchCV(make_svm(list(X.columns), class_weight=class_weight), grid, cv=CV,
                      scoring="accuracy", n_jobs=N_JOBS, refit=False)
    gs.fit(X, y)
    out = {}
    for p, m, s, ft in zip(gs.cv_results_["params"], gs.cv_results_["mean_test_score"],
                           gs.cv_results_["std_test_score"], gs.cv_results_["mean_fit_time"]):
        out[(int(round(np.log2(p["svc__C"]))), int(round(np.log2(p["svc__gamma"]))))] = (m, s, ft)
    return out

def tune(X, y, name, log2C=None, log2G=None, class_weight=None, fixed=None, max_extend=2):
    t0 = time.time()
    tried = {}
    def run(pairs):
        new = [p for p in pairs if p not in tried]
        if new:
            tried.update(_grid_eval(X, y, new, class_weight))
    best = lambda: max(tried, key=lambda k: tried[k][0])
    if fixed is not None:
        run([tuple(fixed)])
    else:
        log2C = LOG2_C_COARSE if log2C is None else log2C
        log2G = LOG2_G_COARSE if log2G is None else log2G
        run(list(itertools.product(log2C, log2G)))
        for _ in range(max_extend):                       # 경계에 걸리면 범위 확장
            c, g = best()
            cs = sorted({k[0] for k in tried}); gs_ = sorted({k[1] for k in tried})
            ext = []
            if c == cs[0]:  ext += [(c - 2, gg) for gg in gs_]
            if c == cs[-1]: ext += [(c + 2, gg) for gg in gs_]
            if g == gs_[0]:  ext += [(cc, g - 2) for cc in cs]
            if g == gs_[-1]: ext += [(cc, g + 2) for cc in cs]
            if not ext: break
            print(f"  [{name}] 최적값 (log2C={c}, log2γ={g})이 경계 → 범위 확장")
            run(ext)
        c, g = best()
        run([(c + dc, g + dg) for dc in (-1, 0, 1) for dg in (-1, 0, 1)])   # 1칸 간격 세밀 탐색
    t_search = time.time() - t0
    c, g = best()
    tab = pd.DataFrame([dict(log2C=k[0], log2gamma=k[1], cv_acc_mean=v[0], cv_acc_std=v[1], fit_time_per_fold=v[2])
                        for k, v in tried.items()]).sort_values("cv_acc_mean", ascending=False).reset_index(drop=True)
    CV_TABLES[name] = tab.assign(experiment=name)
    info = dict(log2C=c, log2gamma=g, best_C=2.0 ** c, best_gamma=2.0 ** g,
                cv_acc=tried[(c, g)][0], cv_std=tried[(c, g)][1], n_evaluated=len(tried), search_time_s=t_search)
    print(f"[{name}] best log2C={c}, log2γ={g} | CV acc = {info['cv_acc']:.4f} ± {info['cv_std']:.4f} "
          f"| 평가한 조합 {len(tried)}개 | 탐색 {t_search:.0f}s")
    if fixed is None:
        print(tab.head(8).round(4).to_string())
    return info

def refit(X, y, info, class_weight=None):
    model = make_svm(list(X.columns), C=info["best_C"], gamma=info["best_gamma"], class_weight=class_weight)
    t0 = time.time(); model.fit(X, y); info["refit_time_s"] = time.time() - t0
    info["n_support"] = int(model[-1].n_support_.sum())
    return model

def timed_predict(model, X):
    t0 = time.time(); p = model.predict(X); return p, time.time() - t0

RESULTS, PREDS = [], {}

## 4. Group-only 함수

- `FIXED`로 준 (log2C, log2γ)로 5-fold 점수만 구한다. 기준으로 전체 109개 조건도 같은 설정으로 다시 계산한다(그룹 제거 실험의 기준값과 같아야 한다).
- **RBF gamma 주의**: 표준화한 특징에서 두 기록 사이 거리²는 특징 수에 비례한다. 109개에 맞춘 gamma를 특징 3개짜리 그룹에 그대로 쓰면 커널이 거의 평평해져(사실상 선형에 가까움) 그룹 자체의 정보보다 낮게 나올 수 있다.
  그래서 요청받은 **고정 조건(주 결과)** 외에, 특징 수 차이만 보정한 **참고 조건**을 함께 기록한다.
  - 참고 조건: γ_그룹 = γ_전체 × (전체 유효 특징 수 ÷ 그룹 유효 특징 수), C는 그대로
  - 유효 특징 수 = 학습 데이터에서 상수가 아닌 특징 수

In [7]:
from sklearn.model_selection import cross_validate

def cv_fixed(X, y, log2C, log2G):
    model = make_svm(list(X.columns), C=2.0 ** log2C, gamma=2.0 ** log2G)
    t0 = time.time()
    r = cross_validate(model, X, y, cv=CV, scoring="accuracy", n_jobs=N_JOBS)
    return r["test_score"].mean(), r["test_score"].std(), time.time() - t0

def n_effective(tr, feats):
    return int((tr[feats].nunique() > 1).sum())

def group_only(df, ycol, fixed, tag):
    tr = df[df.split == "train"]
    c, g = fixed
    n_full = n_effective(tr, FEATS_ALL)
    rows = []
    m, s, t = cv_fixed(tr[FEATS_ALL], tr[ycol], c, g)
    rows.append(dict(group="(전체)", n_features=len(FEATS_ALL), n_effective=n_full, log2C=c, log2gamma=g,
                     cv_acc=m, cv_std=s, log2gamma_scaled=g, cv_acc_scaled=m, cv_std_scaled=s))
    print(f"[{tag}] 전체 {len(FEATS_ALL)}개: CV acc {m:.4f} ± {s:.4f} ({t:.0f}s)")
    for grp, feats in GROUPS.items():
        n_eff = n_effective(tr, feats)
        m, s, t = cv_fixed(tr[feats], tr[ycol], c, g)
        g_sc = g + np.log2(n_full / n_eff)
        m2, s2, t2 = cv_fixed(tr[feats], tr[ycol], c, g_sc)
        rows.append(dict(group=grp, n_features=len(feats), n_effective=n_eff, log2C=c, log2gamma=g,
                         cv_acc=m, cv_std=s, log2gamma_scaled=round(g_sc, 2), cv_acc_scaled=m2, cv_std_scaled=s2))
        print(f"[{tag}] {grp}만 ({len(feats)}개, 유효 {n_eff}개): 고정 {m:.4f} ± {s:.4f} | "
              f"gamma 보정(log2γ={g_sc:.2f}) {m2:.4f} ± {s2:.4f} ({t + t2:.0f}s)")
    return pd.DataFrame(rows)

GROUP_NAMES = {"g1": "개수·비율", "g2": "순서·위치", "g3": "연속 묶음", "g4": "초반·후반 방향", "g5": "시간·속도", "g6": "누적 흐름"}

## 5. Closed-world (95 클래스)

기준선: 무작위 1/95 = 0.0105

In [8]:
FIXED_CLOSED = (6, -6)    # 그룹 제거 실험(Closed-world)과 같은 설정
go_closed = group_only(mon, "label", FIXED_CLOSED, "closed")

[closed] 전체 109개: CV acc 0.8428 ± 0.0051 (26s)
[closed] g1만 (3개, 유효 3개): 고정 0.2201 ± 0.0046 | gamma 보정(log2γ=-0.86) 0.3375 ± 0.0050 (33s)
[closed] g2만 (17개, 유효 17개): 고정 0.4907 ± 0.0063 | gamma 보정(log2γ=-3.36) 0.5618 ± 0.0066 (36s)
[closed] g3만 (6개, 유효 6개): 고정 0.3253 ± 0.0020 | gamma 보정(log2γ=-1.86) 0.4141 ± 0.0069 (34s)
[closed] g4만 (3개, 유효 3개): 고정 0.2099 ± 0.0046 | gamma 보정(log2γ=-0.86) 0.2658 ± 0.0062 (36s)
[closed] g5만 (30개, 유효 28개): 고정 0.5232 ± 0.0028 | gamma 보정(log2γ=-4.08) 0.5876 ± 0.0054 (43s)
[closed] g6만 (50개, 유효 49개): 고정 0.5070 ± 0.0114 | gamma 보정(log2γ=-4.89) 0.6105 ± 0.0105 (43s)


## 6. Open-world 이진

기준선: 전부 mon으로 답하면 학습용 기준 0.655 (15,200 / 23,200)

> **해석 주의**: 원본 7번 패킷 방향 하나로 mon/unmon이 0.9999 갈린다(LR 보고서 6절). g4(처음 30·100개 송신 비율)는 8~13번 패킷 차이를 직접 담으므로, **g4 단독 결과는 수집 흔적의 영향을 크게 받을 수 있다.** 조교님 답변에 따른 특징 사용 조건을 확인한 뒤 해석한다.

In [9]:
FIXED_BINARY = (3, -4)    # 그룹 제거 실험(이진)과 같은 설정
bv = binary_view(mon, unmon) if "binary_view" in globals() else None
if bv is None:
    bv = pd.concat([mon, unmon], ignore_index=True)
    bv["y"] = np.where(bv.dataset == "mon", 1, -1)
go_binary = group_only(bv, "y", FIXED_BINARY, "binary")

/usr/local/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
[binary] 전체 109개: CV acc 0.8907 ± 0.0037 (51s)
/usr/local/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
[binary] g1만 (3개, 유효 3개): 고정 0.6642 ± 0.0009 | gamma 보정(log2γ=1.17) 0.6837 ± 0.0057 (152s)
/usr/local/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
[binary] g2만 (17개, 유효 17개): 고정 0.7161 ± 0.0031 | gamma 보정(log2γ=-1.33) 0.7387 ± 0.0032 (88s)
[binary] g3만 (6개, 유효 6개):

## 7. 그룹 제거 결과와 나란히 보기

- **단독 성능(Group-only)**: 그 그룹만으로 얼마나 구별되는가
- **제거 시 하락(ablation)**: 다른 그룹이 있을 때 그 그룹만의 고유한 기여가 얼마인가
- 단독 성능은 높은데 제거 시 하락이 작으면 → 다른 그룹과 정보가 겹친다
- 단독 성능은 낮은데 제거 시 하락이 크면 → 혼자서는 약하지만 다른 그룹을 보완한다

In [10]:
def side_by_side(go, abl_file):
    out = go.copy()
    out.insert(1, "name", out.group.map(GROUP_NAMES).fillna("전체 109개"))
    path = os.path.join(RESULT_DIR, abl_file)
    if os.path.exists(path):
        abl = pd.read_csv(path).rename(columns={"removed": "group", "delta_vs_full": "ablation_delta"})
        out = out.merge(abl[["group", "ablation_delta"]], on="group", how="left")
    else:
        print(f"{abl_file} 없음 → ablation 열 생략")
    return out

cols = ["group", "name", "n_features", "n_effective", "cv_acc", "cv_std", "cv_acc_scaled", "ablation_delta"]
side_closed = side_by_side(go_closed, "svm_ablation_closed.csv")
side_binary = side_by_side(go_binary, "svm_ablation_binary.csv")
print("[Closed-world]"); print(side_closed[[c for c in cols if c in side_closed]].round(4).to_string(index=False))
print("\n[Open-world 이진]"); print(side_binary[[c for c in cols if c in side_binary]].round(4).to_string(index=False))

[Closed-world]
group     name  n_features  n_effective  cv_acc  cv_std  cv_acc_scaled  ablation_delta
 (전체)  전체 109개         109          106  0.8428  0.0051         0.8428             NaN
   g1    개수·비율           3            3  0.2201  0.0046         0.3375         -0.0010
   g2    순서·위치          17           17  0.4907  0.0063         0.5618         -0.0092
   g3    연속 묶음           6            6  0.3253  0.0020         0.4141         -0.0067
   g4 초반·후반 방향           3            3  0.2099  0.0046         0.2658         -0.0316
   g5    시간·속도          30           28  0.5232  0.0028         0.5876         -0.0199
   g6    누적 흐름          50           49  0.5070  0.0114         0.6105         -0.0449

[Open-world 이진]
group     name  n_features  n_effective  cv_acc  cv_std  cv_acc_scaled  ablation_delta
 (전체)  전체 109개         109          108  0.8907  0.0037         0.8907             NaN
   g1    개수·비율           3            3  0.6642  0.0009         0.6837         -0.0003
   g2    순서

## 8. 저장

In [11]:
side_closed.to_csv(os.path.join(RESULT_DIR, "svm_group_only_closed.csv"), index=False)
side_binary.to_csv(os.path.join(RESULT_DIR, "svm_group_only_binary.csv"), index=False)
print("저장 완료: svm_group_only_closed.csv, svm_group_only_binary.csv")

저장 완료: svm_group_only_closed.csv, svm_group_only_binary.csv
